In [2]:
!pip install -q openai chromadb rank_bm25 sentence-transformers pypdf tiktoken anthropic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 992.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 932.0/932.0 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6

In [3]:
import os
import json
import hashlib
import re
import numpy as np
from typing import List, Dict, Tuple
from google.colab import userdata

# Set your API keys
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [4]:
from pypdf import PdfReader

def read_pdf_content(file_path: str) -> str:
    """Reads a PDF file and extracts its text content."""
    reader = PdfReader(file_path)
    text_content = ""
    for page in reader.pages:
        text_content += page.extract_text() + "\n"
    return text_content.strip()

In [7]:
import os

# List of selected PDF files
pdf_files_to_process = [
    "/content/sample_data/35f78458-fda9-4486-b457-33d418a9eca5.pdf",
    "/content/sample_data/02f12cbd-9474-4623-a1ad-2057885862c1.pdf",
    "/content/sample_data/5d8fbf36-7a03-4037-b2a5-024adb2cb2e5.pdf"
]

# Initialize an empty DOCUMENTS list
DOCUMENTS = []

for file_path in pdf_files_to_process:
    content = read_pdf_content(file_path)
    file_name = os.path.basename(file_path)
    title = file_name.replace(".pdf", "").replace("-", " ").replace("_", " ").title() # Basic title extraction

    DOCUMENTS.append({
        "title": title,
        "source": file_name,
        "content": content
    })

print(f"Successfully loaded {len(DOCUMENTS)} documents from PDF files.")

Successfully loaded 3 documents from PDF files.


In [8]:
print("\n--- Sample of Loaded Documents ---")
for i, doc in enumerate(DOCUMENTS):
    print(f"\nDocument {i+1} - Title: {doc['title']}")
    print(f"  Content (first 200 chars): {doc['content'][:200]}...")


--- Sample of Loaded Documents ---

Document 1 - Title: 35F78458 Fda9 4486 B457 33D418A9Eca5
  Content (first 200 chars): Date: May 18, 2026 
 
To,  
Listing Department 
National Stock Exchange of India Limited 
Exchange Plaza, C-1, G Block, Bandra Kurla Complex, 
Bandra (East), Mumbai - 400 051. 
Symbol: SYRMA 
 
Depart...

Document 2 - Title: 02F12Cbd 9474 4623 A1Ad 2057885862C1
  Content (first 200 chars): Date: September 03, 2025 
 
To,  
Listing Department 
National Stock Exchange of India Limited 
Exchange Plaza, C-1, G Block, Bandra Kurla Complex, Bandra 
(East), Mumbai - 400 051. 
Symbol: SYRMA 
 
...

Document 3 - Title: 5D8Fbf36 7A03 4037 B2A5 024Adb2Cb2E5
  Content (first 200 chars): Date: June 07, 2026  To,  Listing Department National Stock Exchange of India Limited Exchange Plaza, C-1, G Block, Bandra Kurla Complex, Bandra (East), Mumbai - 400 051. Symbol: SYRMA  Department of ...


In [9]:
def chunk_fixed(text: str, chunk_size: int = 3000, overlap: int = 50) -> List[str]:
    """Split text into fixed-size word chunks with overlap."""
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        if chunk.strip():
            chunks.append(chunk.strip())
    return chunks

In [10]:
def chunk_recursive(text: str, max_size: int = 3000) -> List[str]:
    """Split text on natural boundaries: paragraphs, then sentences."""
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]

    chunks = []
    for para in paragraphs:
        words = para.split()
        if len(words) <= max_size:
            chunks.append(para)
        else:
            sentences = para.replace(". ", ".\n").split("\n")
            current, current_len = [], 0
            for sent in sentences:
                sent_len = len(sent.split())
                if current_len + sent_len > max_size and current:
                    chunks.append(" ".join(current))
                    current, current_len = [sent], sent_len
                else:
                    current.append(sent)
                    current_len += sent_len
            if current:
                chunks.append(" ".join(current))

    return [c for c in chunks if len(c.split()) > 10]

In [11]:
sample = DOCUMENTS[0]["content"]

fixed = chunk_fixed(sample, chunk_size=3000, overlap=200)
recursive = chunk_recursive(sample, max_size=3000)

print("=" * 60)
print("FIXED-SIZE CHUNKING")
print("=" * 60)
for i, c in enumerate(fixed[:3]):
    print(f"\nChunk {i+1} ({len(c.split())} words):")
    print(c[:150], "...")

print("\n" + "=" * 60)
print("RECURSIVE CHUNKING")
print("=" * 60)
for i, c in enumerate(recursive[:3]):
    print(f"\nChunk {i+1} ({len(c.split())} words):")
    print(c[:150], "...")

print("\n💡 Fixed cuts mid-thought. Recursive respects paragraph boundaries.")

FIXED-SIZE CHUNKING

Chunk 1 (3000 words):
Date: May 18, 2026 To, Listing Department National Stock Exchange of India Limited Exchange Plaza, C-1, G Block, Bandra Kurla Complex, Bandra (East),  ...

Chunk 2 (3000 words):
PCB and any other projects that you're undertaking right now? Bijay Agrawal: Sure. I'll take this one. So overall, the large project which we are taki ...

Chunk 3 (3000 words):
Sir, if you can throw some more insights in terms of the value -add products and inputs that we're targeting within the industrial segment? And how do ...

RECURSIVE CHUNKING

Chunk 1 (268 words):
Date: May 18, 2026 
 
To,  
Listing Department 
National Stock Exchange of India Limited 
Exchange Plaza, C-1, G Block, Bandra Kurla Complex, 
Bandra  ...

Chunk 2 (3000 words):
Syrma SGS Technology Limited     May 12, 2026    Page 2 of 19    Moderator: Ladies and gentlemen, good day, and welcome to Syrma SGS Technology Limite ...

Chunk 3 (2986 words):
account.   We believe that in the current year, 

In [12]:
from openai import OpenAI
import chromadb

oai = OpenAI()

def get_embeddings(texts: List[str], model: str = "text-embedding-3-small") -> List[List[float]]:
    """Batch embed texts with OpenAI."""
    cleaned = [t.replace("\n", " ").strip() for t in texts]
    resp = oai.embeddings.create(input=cleaned, model=model)
    return [d.embedding for d in resp.data]

def get_embedding(text: str) -> List[float]:
    return get_embeddings([text])[0]

In [13]:
# --- Chunk all documents ---

all_chunks = []
chunk_meta = []

for doc in DOCUMENTS:
    chunks = chunk_recursive(doc["content"], max_size=100)
    for chunk in chunks:
        all_chunks.append(chunk)
        chunk_meta.append({"title": doc["title"], "source": doc["source"]})

print(f"Total chunks: {len(all_chunks)}")
for doc in DOCUMENTS:
    n = sum(1 for m in chunk_meta if m["title"] == doc["title"])
    print(f"  {doc['title']}: {n} chunks")

Total chunks: 1937
  35F78458 Fda9 4486 B457 33D418A9Eca5: 105 chunks
  02F12Cbd 9474 4623 A1Ad 2057885862C1: 1781 chunks
  5D8Fbf36 7A03 4037 B2A5 024Adb2Cb2E5: 51 chunks


In [14]:
# --- Embed and store in ChromaDB ---

chroma = chromadb.Client()

try: chroma.delete_collection("naive_rag")
except: pass

collection = chroma.create_collection("naive_rag", metadata={"hnsw:space": "cosine"})

embs = get_embeddings(all_chunks)

collection.add(
    ids=[f"chunk_{i}" for i in range(len(all_chunks))],
    embeddings=embs,
    documents=all_chunks,
    metadatas=chunk_meta
)

print(f"✅ Stored {len(all_chunks)} chunks in ChromaDB")

✅ Stored 1937 chunks in ChromaDB


In [15]:
def naive_rag(question: str, k: int = 5, verbose: bool = True) -> str:
    results = collection.query(
        query_embeddings=[get_embedding(question)],
        n_results=k
    )

    docs = results["documents"][0]
    metas = results["metadatas"][0]
    dists = results["distances"][0]

    if verbose:
        print(f"\n🔍 Query: '{question}'")
        print(f"\nRetrieved {k} chunks:")
        for i, (d, m, dist) in enumerate(zip(docs, metas, dists)):
            print(f"  [{i+1}] dist={dist:.3f} | {m['title']}")
            print(f"      {d[:80]}...")

    context = "\n".join(
        f"[Source: {m['title']}]\n{d}" for d, m in zip(docs, metas)
    )

    resp = oai.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[{"role": "user", "content": f"""Answer based ONLY on the context below.
If the context doesn't have the answer, say \"I don't have enough information.\"
Cite your sources.

Context:
{context}

Question: {question}

Answer:"""}]
    )

    answer = resp.choices[0].message.content
    if verbose:
        print(f"\n💬 Answer:\n{answer}")
    return answer

In [16]:
naive_rag("What is the business of syrma");


🔍 Query: 'What is the business of syrma'

Retrieved 5 chunks:
  [1] dist=0.356 | 02F12Cbd 9474 4623 A1Ad 2057885862C1
      Syrma Semicon Private Limited 10. Syrma SGS Design and Manufacturing Private Lim...
  [2] dist=0.377 | 02F12Cbd 9474 4623 A1Ad 2057885862C1
      of customer needs, the Company employs market research,  CRM software and digita...
  [3] dist=0.395 | 02F12Cbd 9474 4623 A1Ad 2057885862C1
       Syrma SGS operates as a B2B EMS provider, supplying components and assemblies t...
  [4] dist=0.398 | 02F12Cbd 9474 4623 A1Ad 2057885862C1
      Syrma SGS is benefiting from robust demand across several  key sectors such as a...
  [5] dist=0.399 | 02F12Cbd 9474 4623 A1Ad 2057885862C1
      delivering tailored electronic solutions through advanced  technology platforms....

💬 Answer:
Syrma operates as a B2B Electronics Manufacturing Services (EMS) provider, supplying components and assemblies to Original Equipment Manufacturers (OEMs) and other businesses. They focus on manufa

In [17]:
# --- TEST 2: Easy — single doc, clear answer ---

naive_rag("Can you tell me about syrma Q4 FY26 results");


🔍 Query: 'Can you tell me about syrma Q4 FY26 results'

Retrieved 5 chunks:
  [1] dist=0.283 | 35F78458 Fda9 4486 B457 33D418A9Eca5
      Bijay Agrawal: Good morning, everyone, and welcome to Syrma SGS earnings call fo...
  [2] dist=0.304 | 35F78458 Fda9 4486 B457 33D418A9Eca5
      Q4 FY '26 Earnings Conference Call”  May 12, 2026                    MANAGEMENT:...
  [3] dist=0.313 | 02F12Cbd 9474 4623 A1Ad 2057885862C1
      Boards’ Report Dear Members, Your Directors are pleased to present the 21st Annu...
  [4] dist=0.314 | 02F12Cbd 9474 4623 A1Ad 2057885862C1
      11 Managing Director’s message  Dear Shareholders, FY 2024-25 has been a period ...
  [5] dist=0.320 | 35F78458 Fda9 4486 B457 33D418A9Eca5
      Syrma SGS Technology Limited     May 12, 2026    Page 2 of 19    Moderator: Ladi...

💬 Answer:
In Q4 FY26, Syrma SGS Technology Limited reported consolidated total revenue of INR 1,477 crores, which represents a 56% increase year-on-year and a 16% increase sequentially. This q

In [18]:
naive_rag("What were syrma quarter 4 results");


🔍 Query: 'What were syrma quarter 4 results'

Retrieved 5 chunks:
  [1] dist=0.485 | 35F78458 Fda9 4486 B457 33D418A9Eca5
      Bijay Agrawal: Good morning, everyone, and welcome to Syrma SGS earnings call fo...
  [2] dist=0.500 | 02F12Cbd 9474 4623 A1Ad 2057885862C1
      Boards’ Report Dear Members, Your Directors are pleased to present the 21st Annu...
  [3] dist=0.529 | 02F12Cbd 9474 4623 A1Ad 2057885862C1
      a robust domestic semiconductor  ecosystem and the availability  of skilled tale...
  [4] dist=0.544 | 35F78458 Fda9 4486 B457 33D418A9Eca5
      A very good morning to all. On behalf of Syrma SGS family, we  welcome you all t...
  [5] dist=0.547 | 02F12Cbd 9474 4623 A1Ad 2057885862C1
      An Internal Complaints  Committee has been set up for the purpose. There were  n...

💬 Answer:
In Quarter 4 of 2026, Syrma SGS reported consolidated total revenue of INR 1,477 crores, which represents a 56% increase on a year-on-year basis and a 16% increase sequentially. This quarter w

In [19]:
from rank_bm25 import BM25Okapi

def tokenize(text: str) -> List[str]:
    """Simple whitespace + lowercase tokenizer for BM25."""
    return re.findall(r'\w+', text.lower())

# Build BM25 index over the same chunks
bm25 = BM25Okapi([tokenize(c) for c in all_chunks])

print(f"✅ Built BM25 index over {len(all_chunks)} chunks")

✅ Built BM25 index over 1937 chunks


In [20]:
def reciprocal_rank_fusion(
    semantic: List[Tuple[int, float]],
    keyword: List[Tuple[int, float]],
    k: int = 60
) -> List[Tuple[int, float]]:
    """
    Merge two ranked lists with RRF.
    Simple, effective, no hyperparameters to tune.
    """
    scores = {}
    for rank, (idx, _) in enumerate(semantic):
        scores[idx] = scores.get(idx, 0) + 1 / (k + rank + 1)
    for rank, (idx, _) in enumerate(keyword):
        scores[idx] = scores.get(idx, 0) + 1 / (k + rank + 1)

    # Added print statement to show raw fused scores
    print("\n--- Reciprocal Rank Fusion Scores (Index, Score) ---")
    for idx, score in sorted(scores.items(), key=lambda x: x[1], reverse=True)[:5]:
        print(f"  Chunk Index: {idx}, RRF Score: {score:.4f}")
    print("---------------------------------------------------")

    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

In [21]:
def hybrid_search(question: str, k: int = 5) -> List[Dict]:
    """Combine semantic + BM25 with RRF."""

    # Semantic search (broad, top 20)
    sem = collection.query(query_embeddings=[get_embedding(question)], n_results=20)
    sem_ranked = [
        (int(id.split("_")[1]), dist)
        for id, dist in zip(sem["ids"][0], sem["distances"][0])
    ]

    # BM25 keyword search (top 20)
    scores = bm25.get_scores(tokenize(question))
    bm25_ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:20]

    # Fuse
    fused = reciprocal_rank_fusion(sem_ranked, bm25_ranked)

    return [
        {"chunk": all_chunks[idx], "meta": chunk_meta[idx], "score": sc}
        for idx, sc in fused[:k]
    ]

In [22]:
def hybrid_rag(question: str, k: int = 5, verbose: bool = True) -> str:
    """RAG with hybrid search."""
    results = hybrid_search(question, k)

    if verbose:
        print(f"\n🔍 Query: '{question}'")
        print(f"\nRetrieved {len(results)} chunks (hybrid):")
        for i, r in enumerate(results):
            print(f"  [{i+1}] RRF={r['score']:.4f} | {r['meta']['title']}")
            print(f"      {r['chunk'][:80]}...")

    context = "\n".join(
        f"[Source: {r['meta']['title']}]\n{r['chunk']}" for r in results
    )

    resp = oai.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[{"role": "user", "content": f"""Answer based ONLY on the context below.
If the context doesn't have the answer, say \"I don't have enough information.\"
Cite your sources.

Context:
{context}

Question: {question}

Answer:"""}]
    )

    answer = resp.choices[0].message.content
    if verbose:
        print(f"\n💬 Answer:\n{answer}")
    return answer

In [23]:
hybrid_rag("What is the business of syrma?");


--- Reciprocal Rank Fusion Scores (Index, Score) ---
  Chunk Index: 1295, RRF Score: 0.0164
  Chunk Index: 72, RRF Score: 0.0164
  Chunk Index: 271, RRF Score: 0.0161
  Chunk Index: 7, RRF Score: 0.0161
  Chunk Index: 647, RRF Score: 0.0159
---------------------------------------------------

🔍 Query: 'What is the business of syrma?'

Retrieved 5 chunks (hybrid):
  [1] RRF=0.0164 | 02F12Cbd 9474 4623 A1Ad 2057885862C1
      Syrma Semicon Private Limited 10. Syrma SGS Design and Manufacturing Private Lim...
  [2] RRF=0.0164 | 35F78458 Fda9 4486 B457 33D418A9Eca5
       So we are now migrating to bigger contracts with bigger customers. So it's not ...
  [3] RRF=0.0161 | 02F12Cbd 9474 4623 A1Ad 2057885862C1
      of customer needs, the Company employs market research,  CRM software and digita...
  [4] RRF=0.0161 | 35F78458 Fda9 4486 B457 33D418A9Eca5
      Over to you, sir.  J.S. Gujral: A very warm welcome, ladies and gentlemen, to th...
  [5] RRF=0.0159 | 02F12Cbd 9474 4623 A1Ad 205788

In [24]:
hybrid_rag("Can you tell me about syrma Q4 FY26 results?");


--- Reciprocal Rank Fusion Scores (Index, Score) ---
  Chunk Index: 3, RRF Score: 0.0293
  Chunk Index: 13, RRF Score: 0.0291
  Chunk Index: 86, RRF Score: 0.0164
  Chunk Index: 2, RRF Score: 0.0161
  Chunk Index: 4, RRF Score: 0.0161
---------------------------------------------------

🔍 Query: 'Can you tell me about syrma Q4 FY26 results?'

Retrieved 5 chunks (hybrid):
  [1] RRF=0.0293 | 35F78458 Fda9 4486 B457 33D418A9Eca5
      Syrma SGS Technology Limited     May 12, 2026    Page 2 of 19    Moderator: Ladi...
  [2] RRF=0.0291 | 35F78458 Fda9 4486 B457 33D418A9Eca5
      Bijay Agrawal: Good morning, everyone, and welcome to Syrma SGS earnings call fo...
  [3] RRF=0.0164 | 35F78458 Fda9 4486 B457 33D418A9Eca5
      And post -sharing, we are  expecting it will be -- net PLI would be approximatel...
  [4] RRF=0.0161 | 35F78458 Fda9 4486 B457 33D418A9Eca5
      Q4 FY '26 Earnings Conference Call”  May 12, 2026                    MANAGEMENT:...
  [5] RRF=0.0161 | 35F78458 Fda9 4486 B45

In [25]:
hybrid_rag("What were syrma quarter 4 results?");


--- Reciprocal Rank Fusion Scores (Index, Score) ---
  Chunk Index: 13, RRF Score: 0.0325
  Chunk Index: 5, RRF Score: 0.0310
  Chunk Index: 557, RRF Score: 0.0299
  Chunk Index: 47, RRF Score: 0.0291
  Chunk Index: 286, RRF Score: 0.0161
---------------------------------------------------

🔍 Query: 'What were syrma quarter 4 results?'

Retrieved 5 chunks (hybrid):
  [1] RRF=0.0325 | 35F78458 Fda9 4486 B457 33D418A9Eca5
      Bijay Agrawal: Good morning, everyone, and welcome to Syrma SGS earnings call fo...
  [2] RRF=0.0310 | 35F78458 Fda9 4486 B457 33D418A9Eca5
      A very good morning to all. On behalf of Syrma SGS family, we  welcome you all t...
  [3] RRF=0.0299 | 02F12Cbd 9474 4623 A1Ad 2057885862C1
      An Internal Complaints  Committee has been set up for the purpose. There were  n...
  [4] RRF=0.0291 | 35F78458 Fda9 4486 B457 33D418A9Eca5
      Bijay Agrawal: So when we see, with the higher delivery in this quarter, we have...
  [5] RRF=0.0161 | 02F12Cbd 9474 4623 A1Ad 2057

In [26]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("✅ Loaded cross-encoder reranker")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✅ Loaded cross-encoder reranker


In [27]:
def rerank(question: str, results: List[Dict], top_k: int = 3) -> List[Dict]:
    """Rerank results with cross-encoder."""
    pairs = [(question, r["chunk"]) for r in results]
    scores = reranker.predict(pairs)
    for r, s in zip(results, scores):
        r["rerank_score"] = float(s)
    return sorted(results, key=lambda x: x["rerank_score"], reverse=True)[:top_k]

In [28]:
def hybrid_rag_with_reranker(question: str, verbose: bool = True) -> str:
    """
    Hybrid search (top 10) → Rerank (top 3) → Generate
    (without Anthropic contextualization)
    """
    # Use hybrid_search on the original chunks
    results = hybrid_search(question, k=10)
    top = rerank(question, results, top_k=3)

    if verbose:
        print(f"\n🔍 Query: '{question}'")
        print(f"\nTop 3 after reranking (non-contextualized):")
        for i, r in enumerate(top):
            print(f"  [{i+1}] rerank={r['rerank_score']:.3f} | {r['meta']['title']}")
            print(f"      {r['chunk'][:100]}...")

    context = "\n".join(
        f"[Source: {r['meta']['title']}]\n{r['chunk']}" for r in top
    )

    resp = oai.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[{"role": "user", "content": f"""Answer based ONLY on the context below.
If the context doesn't have the answer, say \"I don't have enough information.\"
Cite your sources.

Context:
{context}

Question: {question}

Answer:"""}]
    )

    answer = resp.choices[0].message.content
    if verbose:
        print(f"\n💬 Answer:\n{answer}")
    return answer

In [29]:
hybrid_rag_with_reranker("What is the business of syrma?");


--- Reciprocal Rank Fusion Scores (Index, Score) ---
  Chunk Index: 1295, RRF Score: 0.0164
  Chunk Index: 72, RRF Score: 0.0164
  Chunk Index: 271, RRF Score: 0.0161
  Chunk Index: 7, RRF Score: 0.0161
  Chunk Index: 647, RRF Score: 0.0159
---------------------------------------------------

🔍 Query: 'What is the business of syrma?'

Top 3 after reranking (non-contextualized):
  [1] rerank=4.860 | 02F12Cbd 9474 4623 A1Ad 2057885862C1
      of customer needs, the Company employs market research,  CRM software and digital marketing. By stay...
  [2] rerank=4.860 | 02F12Cbd 9474 4623 A1Ad 2057885862C1
       Syrma SGS operates as a B2B EMS provider, supplying components and assemblies to OEM and other busi...
  [3] rerank=4.168 | 02F12Cbd 9474 4623 A1Ad 2057885862C1
        Syrma SGS operates as a B2B EMS provider, supplying components and assemblies to OEM and other bus...

💬 Answer:
Syrma operates as a B2B EMS (Electronics Manufacturing Services) provider, supplying components and ass

In [30]:
hybrid_rag_with_reranker("Can you tell me about syrma Q4 FY26 results?");


--- Reciprocal Rank Fusion Scores (Index, Score) ---
  Chunk Index: 3, RRF Score: 0.0293
  Chunk Index: 13, RRF Score: 0.0291
  Chunk Index: 86, RRF Score: 0.0164
  Chunk Index: 2, RRF Score: 0.0161
  Chunk Index: 4, RRF Score: 0.0161
---------------------------------------------------

🔍 Query: 'Can you tell me about syrma Q4 FY26 results?'

Top 3 after reranking (non-contextualized):
  [1] rerank=4.576 | 35F78458 Fda9 4486 B457 33D418A9Eca5
      Aniruddha Joshi from ICICI Securities. Thank you, and  over to you, Mr. Joshi.  Aniruddha Joshi: Yes...
  [2] rerank=2.496 | 35F78458 Fda9 4486 B457 33D418A9Eca5
      Syrma SGS Technology Limited     May 12, 2026    Page 2 of 19    Moderator: Ladies and gentlemen, go...
  [3] rerank=2.297 | 35F78458 Fda9 4486 B457 33D418A9Eca5
      Q4 FY '26 Earnings Conference Call”  May 12, 2026                    MANAGEMENT: MR. J. S. GUJRAL – ...

💬 Answer:
I don't have enough information.


In [31]:
hybrid_rag_with_reranker("What were syrma quarter 4 results?");


--- Reciprocal Rank Fusion Scores (Index, Score) ---
  Chunk Index: 13, RRF Score: 0.0325
  Chunk Index: 5, RRF Score: 0.0310
  Chunk Index: 557, RRF Score: 0.0299
  Chunk Index: 47, RRF Score: 0.0291
  Chunk Index: 286, RRF Score: 0.0161
---------------------------------------------------

🔍 Query: 'What were syrma quarter 4 results?'

Top 3 after reranking (non-contextualized):
  [1] rerank=5.820 | 35F78458 Fda9 4486 B457 33D418A9Eca5
      Bijay Agrawal: Good morning, everyone, and welcome to Syrma SGS earnings call for quarter 4 and full...
  [2] rerank=3.580 | 35F78458 Fda9 4486 B457 33D418A9Eca5
      Bijay Agrawal: So when we see, with the higher delivery in this quarter, we have done almost INR1,50...
  [3] rerank=3.180 | 35F78458 Fda9 4486 B457 33D418A9Eca5
      A very good morning to all. On behalf of Syrma SGS family, we  welcome you all to Syrma SGS Quarter ...

💬 Answer:
Syrma SGS reported consolidated total revenue of INR 1,477 crores for quarter 4 of 2026, which repres

In [32]:
def compare_rag_approaches(question: str):
    print(f"\n--- Comparing RAG Approaches for: '{question}' ---")
    print("\n--- Naive RAG ---")
    naive_answer = naive_rag(question, verbose=True)

    print("\n--- Hybrid RAG with Reranker ---")
    hybrid_reranked_answer = hybrid_rag_with_reranker(question, verbose=True)

    print("\n--- Comparison Summary ---")
    print("Naive RAG Answer:", naive_answer)
    print("Hybrid RAG with Reranker Answer:", hybrid_reranked_answer)

Now, let's use the comparison function with a sample query to see the difference.

In [33]:
compare_rag_approaches("What is the business of syrma?")


--- Comparing RAG Approaches for: 'What is the business of syrma?' ---

--- Naive RAG ---

🔍 Query: 'What is the business of syrma?'

Retrieved 5 chunks:
  [1] dist=0.362 | 02F12Cbd 9474 4623 A1Ad 2057885862C1
      Syrma Semicon Private Limited 10. Syrma SGS Design and Manufacturing Private Lim...
  [2] dist=0.376 | 02F12Cbd 9474 4623 A1Ad 2057885862C1
      of customer needs, the Company employs market research,  CRM software and digita...
  [3] dist=0.388 | 02F12Cbd 9474 4623 A1Ad 2057885862C1
       Syrma SGS operates as a B2B EMS provider, supplying components and assemblies t...
  [4] dist=0.393 | 02F12Cbd 9474 4623 A1Ad 2057885862C1
      Name of the Listed Entity Syrma SGS Technology Limited  3. Year of incorporation...
  [5] dist=0.397 | 02F12Cbd 9474 4623 A1Ad 2057885862C1
        Syrma SGS operates as a B2B EMS provider, supplying components and assemblies ...

💬 Answer:
Syrma operates as a B2B EMS (Electronics Manufacturing Services) provider, supplying components and asse

In [34]:
compare_rag_approaches("Can you tell me about syrma Q4 FY26 results?")


--- Comparing RAG Approaches for: 'Can you tell me about syrma Q4 FY26 results?' ---

--- Naive RAG ---

🔍 Query: 'Can you tell me about syrma Q4 FY26 results?'

Retrieved 5 chunks:
  [1] dist=0.279 | 35F78458 Fda9 4486 B457 33D418A9Eca5
      Bijay Agrawal: Good morning, everyone, and welcome to Syrma SGS earnings call fo...
  [2] dist=0.305 | 35F78458 Fda9 4486 B457 33D418A9Eca5
      Q4 FY '26 Earnings Conference Call”  May 12, 2026                    MANAGEMENT:...
  [3] dist=0.310 | 02F12Cbd 9474 4623 A1Ad 2057885862C1
      Boards’ Report Dear Members, Your Directors are pleased to present the 21st Annu...
  [4] dist=0.319 | 35F78458 Fda9 4486 B457 33D418A9Eca5
      Syrma SGS Technology Limited     May 12, 2026    Page 2 of 19    Moderator: Ladi...
  [5] dist=0.321 | 02F12Cbd 9474 4623 A1Ad 2057885862C1
      11 Managing Director’s message  Dear Shareholders, FY 2024-25 has been a period ...

💬 Answer:
In Q4 FY26, Syrma SGS reported consolidated total revenue of INR 1,477 crore

In [35]:
def debug_reranker_process(question: str):
    print(f"\n--- Debugging Reranker Process for: '{question}' ---")

    # Step 1: Perform hybrid search
    print("\n--- Hybrid Search Results (Top 10) ---")
    hybrid_results = hybrid_search(question, k=10)
    for i, r in enumerate(hybrid_results):
        print(f"  [{i+1}] RRF={r['score']:.4f} | {r['meta']['title']}")
        print(f"      {r['chunk'][:100]}...")

    # Step 2: Perform reranking on hybrid search results
    print("\n--- Reranking Results (Top 3 from Hybrid) ---")
    reranked_results = rerank(question, hybrid_results, top_k=3)
    for i, r in enumerate(reranked_results):
        print(f"  [{i+1}] rerank={r['rerank_score']:.3f} | {r['meta']['title']}")
        print(f"      {r['chunk'][:100]}...")

    print("\n--- Final Context used by LLM after Reranking ---")
    context = "\n".join(
        f"[Source: {r['meta']['title']}]\n{r['chunk']}" for r in reranked_results
    )
    print(context)

Now, let's use this debugging function for the query where `hybrid_rag_with_reranker` failed: "Can you tell me about syrma Q4 FY26 results?"

In [36]:
debug_reranker_process("Can you tell me about syrma Q4 FY26 results?")


--- Debugging Reranker Process for: 'Can you tell me about syrma Q4 FY26 results?' ---

--- Hybrid Search Results (Top 10) ---

--- Reciprocal Rank Fusion Scores (Index, Score) ---
  Chunk Index: 3, RRF Score: 0.0293
  Chunk Index: 13, RRF Score: 0.0291
  Chunk Index: 86, RRF Score: 0.0164
  Chunk Index: 2, RRF Score: 0.0161
  Chunk Index: 4, RRF Score: 0.0161
---------------------------------------------------
  [1] RRF=0.0293 | 35F78458 Fda9 4486 B457 33D418A9Eca5
      Syrma SGS Technology Limited     May 12, 2026    Page 2 of 19    Moderator: Ladies and gentlemen, go...
  [2] RRF=0.0291 | 35F78458 Fda9 4486 B457 33D418A9Eca5
      Bijay Agrawal: Good morning, everyone, and welcome to Syrma SGS earnings call for quarter 4 and full...
  [3] RRF=0.0164 | 35F78458 Fda9 4486 B457 33D418A9Eca5
      And post -sharing, we are  expecting it will be -- net PLI would be approximately INR38 crores relat...
  [4] RRF=0.0161 | 35F78458 Fda9 4486 B457 33D418A9Eca5
      Q4 FY '26 Earnings Confe